# N-gram Practice: RAG Text

This notebook loads the supplied corpus, builds unigram and bigram models, estimates the requested conditional probabilities, and generates RAG-related text with unigram sampling and a bigram Markov chain.

In [ ]:
from collections import Counter, defaultdict
from pathlib import Path
from zipfile import ZipFile
import random
import re

ZIP_PATH = Path(r'C:/Users/ACER/Downloads/22_05_2026___003bfd40afee336da814a9cb1e8000a0.zip')
with ZipFile(ZIP_PATH) as zf:
    txt_name = next(name for name in zf.namelist() if name.endswith('.txt'))
    corpus = zf.read(txt_name).decode('utf-8')

# Keep identifiers such as vector-db and clean_text as single tokens.
TOKEN_RE = re.compile(r"[a-z0-9_]+(?:[-'][a-z0-9_]+)*", re.IGNORECASE)
tokens = [m.group(0).lower() for m in TOKEN_RE.finditer(corpus)]
sentences = [[m.group(0).lower() for m in TOKEN_RE.finditer(line)]
             for line in corpus.splitlines()]
sentences = [s for s in sentences if s]
print(f'Sentences: {len(sentences):,}')
print(f'Tokens: {len(tokens):,}')
print(f'Vocabulary: {len(set(tokens)):,}')

## Task 1: Unigram model

For a token `w`, the maximum-likelihood unigram estimate is `P(w) = count(w) / total_tokens`.

In [ ]:
unigram_counts = Counter(tokens)
total_tokens = len(tokens)
unigram_probabilities = {word: count / total_tokens
                       for word, count in unigram_counts.items()}

top_20 = [(word, count, unigram_probabilities[word])
          for word, count in unigram_counts.most_common(20)]
for rank, (word, count, probability) in enumerate(top_20, start=1):
    print(f'{rank:>2}. {word:<20} count={count:<4} P={probability:.6f}')

## Task 2: Bigram model

For a bigram `(w1, w2)`, the unsmoothed estimate is `P(w2 | w1) = count(w1, w2) / count(w1)`. Add-one smoothing is also shown so unseen bigrams receive a small non-zero probability.

In [ ]:
bigram_counts = defaultdict(Counter)
for sentence in sentences:
    for previous, current in zip(sentence, sentence[1:]):
        bigram_counts[previous][current] += 1

def bigram_probability(current, previous, alpha=0.0):
    denominator = unigram_counts[previous] + alpha * len(unigram_counts)
    numerator = bigram_counts[previous][current] + alpha
    return numerator / denominator if denominator else 0.0

queries = [('generation', 'retrieval'), ('search', 'semantic'), ('models', 'language')]
for current, previous in queries:
    mle = bigram_probability(current, previous)
    laplace = bigram_probability(current, previous, alpha=1.0)
    print(f'P({current} | {previous}) = MLE {mle:.6f}, add-one {laplace:.6f}')

## Task 3: Sentence generation

The unigram generator ignores word order. The bigram generator chooses the next word from the learned continuation distribution. If a word has no observed continuation, it falls back to unigram sampling.

In [ ]:
def weighted_choice(counter):
    words = list(counter.keys())
    weights = list(counter.values())
    return random.choices(words, weights=weights, k=1)[0]

def generate_unigram(length=20, seed=42):
    random.seed(seed)
    return ' '.join(weighted_choice(unigram_counts) for _ in range(length))

def generate_bigram(length=20, seed=42):
    random.seed(seed)
    current = weighted_choice(unigram_counts)
    output = [current]
    for _ in range(length - 1):
        next_words = bigram_counts.get(current)
        current = weighted_choice(next_words or unigram_counts)
        output.append(current)
    return ' '.join(output)

for seed in range(3):
    print(f'Unigram {seed + 1}: {generate_unigram(seed=seed)}')
    print(f'Bigram  {seed + 1}: {generate_bigram(seed=seed)}')
    print()

## Interpretation

Unigram text usually looks incoherent because it models only individual-token frequency. Bigram text preserves local two-token patterns such as `retrieval quality`, `vector databases`, and `semantic retrieval`, but it still lacks the longer context captured by trigram or neural language models.